# 🚀 Stock Bot GRPO Training on Google Colab

이 노트북은 한국 주식 시장 스캘핑을 위한 GRPO(Group Relative Policy Optimization) 강화학습 에이전트를 Google Colab에서 훈련합니다.

## 📋 Overview
- **목적**: 임베딩 기반 스캘핑 정책 학습
- **알고리즘**: GRPO (Group Relative Policy Optimization)
- **데이터**: DuckDB 정규화 데이터
- **훈련 시간**: 약 6-12시간 (GPU 사용시)

## 🔧 Environment Setup

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install duckdb numpy pandas tqdm matplotlib seaborn tensorboard

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content')
!mkdir -p models logs data

EMBEDDING_MODEL_PATH = '/content/drive/MyDrive/ColabData/models/stockbot/autoencoder/20251012_013148/model.pt'
DB_PATH = '/content/drive/MyDrive/ColabData/datasets/stockbot/datasets_norm_all.duckdb'

print("\n📁 Checking required files...")
print(f"Embedding model: {'✅' if os.path.exists(EMBEDDING_MODEL_PATH) else '❌'} {EMBEDDING_MODEL_PATH}")
print(f"Database: {'✅' if os.path.exists(DB_PATH) else '❌'} {DB_PATH}")

## 📦 GitHub Repository

In [ ]:
from google.colab import userdata

try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 Using GitHub token")
except:
    print("⚠️ GITHUB_TOKEN not found")
    use_token = False

if not os.path.exists('/content/stock-bot2'):
    print("📥 Cloning repository...")
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git /content/stock-bot2
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git /content/stock-bot2
else:
    print("📁 Updating repository...")
    %cd /content/stock-bot2
    !git pull

%cd /content/stock-bot2
import sys
sys.path.append('/content/stock-bot2')
print(f"✅ Ready! Current directory: {os.getcwd()}")

## ⚙️ Training Configuration

In [ ]:
CONFIG = {
    'embedding_model': EMBEDDING_MODEL_PATH,
    'db': DB_PATH,
    'table': 'datasets',
    'out': '/content/models/grpo_scalping',
    'seq_len': 60,
    'transaction_cost_rate': 0.00215,
    'max_holding_time': 60.0,
    'holding_penalty_rate': 0.001,
    'quick_exit_threshold': 1.5,
    'quick_exit_penalty': 0.01,
    'episodes_per_group': 12,
    'num_groups': 4,
    'total_timesteps': 1000000,
    'hidden_dim': 256,
    'learning_rate': 3e-4,
    'gamma': 0.99,
    'clip_epsilon': 0.2,
    'kl_target': 0.01,
    'entropy_coef': 0.01,
    'value_coef': 0.5,
    'max_grad_norm': 0.5,
    'checkpoint_interval': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🖥️ GPU Memory: {gpu_memory:.1f} GB")
    if gpu_memory < 12:
        CONFIG.update({'episodes_per_group': 8, 'num_groups': 3, 'total_timesteps': 500000})
        print("⚠️ Adjusted for limited GPU")
    elif gpu_memory > 25:
        CONFIG.update({'episodes_per_group': 16, 'num_groups': 6, 'total_timesteps': 2000000})
        print("🚀 Adjusted for high-end GPU")

print("\n📋 Configuration:")
for k, v in CONFIG.items():
    print(f"  {k:25s}: {v}")

## 🚀 GRPO Training

In [ ]:
cmd = f"""python -m ai_trader.grpo.train_grpo \\
  --embedding-model \"{CONFIG['embedding_model']}\" \\
  --db \"{CONFIG['db']}\" \\
  --table {CONFIG['table']} \\
  --out {CONFIG['out']} \\
  --seq-len {CONFIG['seq_len']} \\
  --episodes-per-group {CONFIG['episodes_per_group']} \\
  --num-groups {CONFIG['num_groups']} \\
  --total-timesteps {CONFIG['total_timesteps']} \\
  --quick-exit-threshold {CONFIG['quick_exit_threshold']} \\
  --quick-exit-penalty {CONFIG['quick_exit_penalty']} \\
  --transaction-cost-rate {CONFIG['transaction_cost_rate']} \\
  --max-holding-time {CONFIG['max_holding_time']} \\
  --holding-penalty-rate {CONFIG['holding_penalty_rate']} \\
  --hidden-dim {CONFIG['hidden_dim']} \\
  --learning-rate {CONFIG['learning_rate']} \\
  --gamma {CONFIG['gamma']} \\
  --clip-epsilon {CONFIG['clip_epsilon']} \\
  --kl-target {CONFIG['kl_target']} \\
  --entropy-coef {CONFIG['entropy_coef']} \\
  --value-coef {CONFIG['value_coef']} \\
  --max-grad-norm {CONFIG['max_grad_norm']} \\
  --checkpoint-interval {CONFIG['checkpoint_interval']} \\
  --device {CONFIG['device']}"""

print("🚀 Training Command:")
print("=" * 70)
print(cmd)
print("=" * 70)

In [ ]:
!{cmd}

## 📊 TensorBoard

In [ ]:
%load_ext tensorboard
tensorboard_dir = f"{CONFIG['out']}/tensorboard_logs"
print(f"📊 TensorBoard: {tensorboard_dir}")
%tensorboard --logdir {tensorboard_dir}

## 💾 Save to Google Drive

In [ ]:
from datetime import datetime
from pathlib import Path

drive_model_path = '/content/drive/MyDrive/stock_bot_models'
!mkdir -p "{drive_model_path}"

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f'grpo_colab_{timestamp}'
model_dir = f'{drive_model_path}/{model_name}'
!mkdir -p "{model_dir}"

print(f"💾 Saving to: {model_dir}")

!cp -r {CONFIG['out']}/checkpoints "{model_dir}/"
!cp -r {CONFIG['out']}/tensorboard_logs "{model_dir}/"

import json
model_info = {'model_name': model_name, 'timestamp': timestamp, 'config': CONFIG}
with open(f'{model_dir}/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"✅ Model saved: {model_dir}")